# py-Monocle3 — R⇄Py parity inheritance

`pymonocle3` is a thin wrapper around [Bio-Babel/Monocle3-python](https://github.com/Bio-Babel/Monocle3-python), which is itself a comprehensive Python port of R monocle3 (Cao et al. *Nature* 2019).

This notebook documents what that inheritance means for omicverse users.

## 1. Wrapper identity

In [1]:
import pymonocle3, monocle3
print(f'pymonocle3 v{pymonocle3.__version__}  (upstream: monocle3-python v{pymonocle3.__upstream_version__})')
# Every pymonocle3 function is the same object as the upstream monocle3 function
for name in ['preprocess_cds','align_cds','reduce_dimension','cluster_cells',
             'learn_graph','order_cells','plot_cells','graph_test','top_markers']:
    assert getattr(pymonocle3, name) is getattr(monocle3, name)
print('✓ all public functions identical to monocle3-python')

pymonocle3 v0.1.0  (upstream: monocle3-python v1.4.26.post1)
✓ all public functions identical to monocle3-python


## 2. End-to-end smoke pipeline on tiny synthetic

In [2]:
import numpy as np
import anndata as ad
import pymonocle3 as m3

rng = np.random.RandomState(0)
n, g = 200, 150
pt = np.linspace(0, 1, n)
counts = rng.poisson(2.0, (n, g)).astype(np.float32)
# Add gradient in the first 30 genes
for c in range(n):
    counts[c, :30] += rng.poisson(10 * pt[c], 30)
adata = ad.AnnData(X=counts)
adata.obs_names = [f'c{i}' for i in range(n)]
adata.var_names = [f'g{i}' for i in range(g)]
m3.estimate_size_factors(adata)
m3.preprocess_cds(adata, num_dim=20)
m3.reduce_dimension(adata, max_components=2, umap_min_dist=0.1)
m3.cluster_cells(adata)
m3.learn_graph(adata)
print('UMAP shape:', adata.obsm['X_umap'].shape)
print('Clusters:', np.unique(m3.clusters(adata)))
print('Principal graph nodes:', adata.uns.get('principal_graph_nodes', {}).keys() if 'principal_graph_nodes' in adata.uns else 'see adata.uns')

/scratch/users/steorra/env/omicdev/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/scratch/users/steorra/env/omicdev/lib/python3.10/site-packages/umap/umap_.py:2021: UserWarning: precomputed_knn[2] (knn_search_index) is not an NNDescent object: transforming new data with transform will be unavailable.
  warn(


UMAP shape: (200, 2)
Clusters: ['1']
Principal graph nodes: see adata.uns


## 3. Parity guarantees

Because `pymonocle3` re-exports `monocle3-python` verbatim, R⇄Py parity is inherited from upstream. Per the Bio-Babel reconstruction report:

| Function family | Status |
|---|---|
| `preprocess_cds` (size-factor norm + truncated PCA) | ✅ 1:1 |
| `reduce_dimension` (UMAP via umap-learn) | ✅ 1:1 |
| `cluster_cells` (Leiden via leidenalg) | ✅ 1:1 |
| `learn_graph` (SimplePPT principal graph) | ✅ 1:1 |
| `order_cells` (pseudotime by graph distance) | ✅ 1:1 |
| `graph_test` (Moran's I per gene) | ✅ 1:1 |
| `plot_cells` (ggplot2-python rendering) | ✅ 1:1 |

See [Bio-Babel/Monocle3-python](https://github.com/Bio-Babel/Monocle3-python) for the full R reconstruction tutorials reproducing R monocle3 vignettes on the *C. elegans* embryo and L2 datasets.